# 23-07 · Разбор аргументов командной строки

Практика к разделу [«pyproject.toml и установка проекта»](../../site/chapters/glava-23/23-05-pyproject-toml.html) и разделу [«Командная строка SafeSort»](../../site/chapters/glava-23/23-06-komandnaya-stroka.html). Настоящий файл — `projects/python/safesort/src/safesort/cli.py`.

## Цель

Собрать парсер, который повторяет форму настоящего `safesort.cli.build_parser()`: пять подкоманд, у каждой — позиционный аргумент `root`, а `undo` умеет обходиться и вовсе без аргумента.

## Рабочий пример

In [1]:
import argparse
from pathlib import Path


def build_parser():
    parser = argparse.ArgumentParser(
        prog="safesort",
        description=(
            "SafeSort: a safe, non-destructive file organizer. "
            "scan/plan/duplicates never modify anything; only 'apply' moves files."
        ),
    )
    subparsers = parser.add_subparsers(dest="command", required=True)

    for name in ("scan", "plan", "apply", "duplicates"):
        sub = subparsers.add_parser(name)
        sub.add_argument("root", type=Path)

    undo_parser = subparsers.add_parser("undo")
    undo_parser.add_argument("root", type=Path, nargs="?", default=Path("."))

    return parser


parser = build_parser()

args_scan = parser.parse_args(["scan", "/home/anna/Downloads"])
args_apply = parser.parse_args(["apply", "/home/anna/Downloads"])
args_undo_default = parser.parse_args(["undo"])
args_undo_explicit = parser.parse_args(["undo", "/home/anna/Downloads"])

print(args_scan)
print(args_undo_default)

Namespace(command='scan', root=PosixPath('/home/anna/Downloads'))
Namespace(command='undo', root=PosixPath('.'))


## Проверка результата

In [2]:
assert args_scan.command == "scan"
assert args_scan.root == Path("/home/anna/Downloads")
assert args_apply.command == "apply"
assert args_undo_default.command == "undo"
assert args_undo_default.root == Path(".")
assert args_undo_explicit.root == Path("/home/anna/Downloads")
print("Верно: пять подкоманд разобраны, а undo получает root по умолчанию, если он не передан.")

Верно: пять подкоманд разобраны, а undo получает root по умолчанию, если он не передан.


## Эксперимент — без подкоманды argparse сам сообщает об ошибке

In [3]:
import contextlib
import io

buffer = io.StringIO()
try:
    with contextlib.redirect_stderr(buffer):
        parser.parse_args([])
    kod_zaversheniya = None
except SystemExit as exc:
    kod_zaversheniya = exc.code

print("Код завершения:", kod_zaversheniya)
assert kod_zaversheniya is not None and kod_zaversheniya != 0
print("Верно: dest=\"command\", required=True сам формирует понятную ошибку без ручных проверок.")

Код завершения: 2
Верно: dest="command", required=True сам формирует понятную ошибку без ручных проверок.


## Задание ★ Базовая практика

Разберите `["duplicates", "/home/anna/Photos"]` и проверьте, что команда — duplicates, а root — правильный путь.

In [4]:
zadanie_args = parser.parse_args(["duplicates", "/home/anna/Photos"])

assert zadanie_args.command == "duplicates"
assert zadanie_args.root == Path("/home/anna/Photos")
print("Верно: duplicates разобрана с правильным путём.")

Верно: duplicates разобрана с правильным путём.
